# TypeDict

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add


class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state['cur_id']
    return {
        'logs': ['node_1 运行完毕'],
        'cur_id': pre_id + ', node_1'
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state['cur_id']
    return {
        'logs': ['node_2 运行完毕'],
        'cur_id': pre_id + ', node_2'
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)

builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', END)

graph = builder.compile()

res = graph.invoke({'cur_id': 'start'})
print(res)


{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


# dataclass

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from dataclasses import dataclass
from operator import add


@dataclass
class OverAllState:
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_1 运行完毕'],
        'cur_id': pre_id + ', node_1'
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_2 运行完毕'],
        'cur_id': pre_id + ', node_2'
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)

builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', END)

graph = builder.compile()

res = graph.invoke({'cur_id': 'start'})
print(res)


{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


# pydantic

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from pydantic import BaseModel
from operator import add


class OverAllState(BaseModel):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_1 运行完毕'],
        'cur_id': pre_id + ', node_1'
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_2 运行完毕'],
        'cur_id': pre_id + ', node_2'
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)

builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', END)

graph = builder.compile()

res = graph.invoke({'cur_id': 'start'})
print(res)

{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


# State Reducer

In [5]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from pydantic import BaseModel


def my_logs_reducer(old: list[str], new: list[str]) -> list[str]:
    print(f'[INFO]my_logs_reducer 合并操作 {old} + {new}')
    return old + new


class OverAllState(BaseModel):
    logs: Annotated[list[str], my_logs_reducer]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_1 运行完毕'],
        'cur_id': pre_id + ', node_1'
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_2 运行完毕'],
        'cur_id': pre_id + ', node_2'
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)

builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', END)

graph = builder.compile()

res = graph.invoke({'cur_id': 'start'})
print(res)

[INFO]my_logs_reducer 合并操作 [] + ['node_1 运行完毕']
[INFO]my_logs_reducer 合并操作 ['node_1 运行完毕'] + ['node_2 运行完毕']
{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


# Overwrite绕过Reducer

In [10]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from pydantic import BaseModel
from langgraph.types import Overwrite


def my_logs_reducer(old: list[str], new: list[str]) -> list[str]:
    return old + new

def my_cur_id_reducer(old: str, new: str) -> str:
    return new

class OverAllState(BaseModel):
    logs: Annotated[list[str], my_logs_reducer]
    cur_id: Annotated[str, my_cur_id_reducer]

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_1 运行完毕'],
        'cur_id': pre_id + ', node_1'
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': ['node_2 运行完毕'],
        'cur_id': pre_id + ', node_2'
    }

def node_3(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        'logs': Overwrite(['node3 运行完毕，覆盖之前的运行日志！']),
        'cur_id': pre_id + ', node_3'
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)

builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', 'node_3')
builder.add_edge('node_3', END)

graph = builder.compile()

res = graph.invoke({'cur_id': 'start'})
print(res)

{'logs': ['node3 运行完毕，覆盖之前的运行日志！'], 'cur_id': 'start, node_1, node_2, node_3'}


# Multi Schema

In [22]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class InputState(TypedDict):
    username: str

class OutputState(TypedDict):
    graph_output: str

class OverAllState(TypedDict):
    nickname: str
    username: str
    graph_output: str

class PrivateState(TypedDict):
    greeting: str

def node_1(state: InputState) -> OverAllState:
    # 向全局状态写入数据
    return {
        "nickname": "Dear " + state["username"]
    }

def node_2(state: OverAllState) -> PrivateState:
    # 从全局状态读取数据，写入私有状态
    return {
        "greeting": state["nickname"] + ", 早上好~"
    }

def node_3(state: PrivateState) -> OutputState:
    # 从私有状态读取数据，写入输出状态
    return {
        "graph_output": state["greeting"] + " 很高兴认识你！"
    }

builder = StateGraph(OverAllState, input_schema=InputState, output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
print(graph.invoke({"username":"小黄"}))

{'graph_output': 'Dear 小黄, 早上好~ 很高兴认识你！'}


# MessagesState

In [25]:
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage
from rich import print as rprint

from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model='deepseek-flash',
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

class OverAllState(MessagesState):
    username: str
    output: str

def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是 " + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()
response = graph.invoke({"username": "小林"})
rprint(response)

{
    'messages': [
        HumanMessage(
            content='你好，我是 小林',
            additional_kwargs={},
            response_metadata={},
            id='ccef307c-4c24-4e29-a5a1-f376934a6436'
        ),
        AIMessage(
            content='你好，小林！很高兴认识你。我是 DeepSeek，一个由深度求索公司创造的 AI 
助手。\n\n有什么我可以帮你的吗？无论是回答问题、提供建议、聊天，还是帮你处理文字或分析文件，我都很乐意帮忙。请随时
告诉我你想聊什么或需要什么帮助！😊',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 64,
                    'prompt_tokens': 9,
                    'total_tokens': 73,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 9
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '61724590-805a-4b19-a8fb-5e25f88f24e2',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a16438-06ba-7463-85ad-ed936790b603-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 9,
                'output_tokens': 64,
                'total_tokens': 73,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    'username': '小林',
    'output': '你好，小林！很高兴认识你。我是 DeepSeek，一个由深度求索公司创造的 AI 
助手。\n\n有什么我可以帮你的吗？无论是回答问题、提供建议、聊天，还是帮你处理文字或分析文件，我都很乐意帮忙。请随时
告诉我你想聊什么或需要什么帮助！😊'
}

In [ ]:
from langchain.agents import create_agent